In [1]:
!pip install easydict

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
!pip install tritonclient[all]
!pip install tqdm

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [3]:
import numpy as np
import time
import tritonclient.http as tritonhttpclient
import tritonclient.grpc as tritongrpcclient
from tqdm import tqdm
import argparse
import gevent.ssl
import tritonclient.grpc as grpcclient
import tritonclient.http as httpclient
from tritonclient.utils import InferenceServerException
import easydict
import argparse
import numpy as np
import sys
from builtins import range
from ctypes import *
import tritonclient.grpc as grpcclient
from tritonclient import utils
import tritonclient.utils.shared_memory as shm



In [4]:
http_url = 'localhost:8000'
grpc_url = 'localhost:8001'


verbose = False
concurrency = 32
model_version = '1'
triton_http_client = tritonhttpclient.InferenceServerClient(url=http_url, verbose=verbose, concurrency=concurrency)
triton_grpc_client = tritongrpcclient.InferenceServerClient(url=grpc_url, verbose=verbose)


# input_dtype = 'float32'

In [5]:
batch_size=2048
D=1024
model_path='models/simple-trt-model-FP16/1/model.savedmodel'
model_path_copy='models/simple-trt-model-FP16-copy/1/model.savedmodel'
class iterator(object):

    def __init__(self, B=batch_size, D=1024):
        self.B = B # batch size
        self.D = D # dimension

    def __iter__(self):
        self.i = 0
        return self

    def __next__(self):
        output = np.float16(np.random.uniform(size=(self.B, self.D)))

        return output

In [6]:
# !mkdir -p models/simple-tensorflow-model-test/
# !mkdir -p models/simple-tensorflow-model-test/1/
!mkdir -p models/simple-trt-model-FP16/
!mkdir -p models/simple-trt-model-FP16/1/
!mkdir -p models/simple-trt-model-FP16-copy/
!mkdir -p models/simple-trt-model-FP16-copy/1/

In [7]:
import tensorflow as tf
from tensorflow.python.compiler.tensorrt import trt_convert as trt

class WrappedModel(tf.Module):
    def __init__(self):
        super(WrappedModel, self).__init__()
        tf.config.optimizer.set_jit(True)
        self.model = tf.keras.Sequential()
        self.model.add(tf.keras.layers.Dense(64, input_shape=(D,)))
        self.model.add(tf.keras.layers.Dense(32))
        self.model.add(tf.keras.layers.Dense(1))
        self.model.compile(optimizer='sgd', loss='mse')
    @tf.function
    def __call__(self, x):
        return self.model(x)
    
model = WrappedModel()
call = model.__call__.get_concrete_function(tf.TensorSpec([None,None], 
                                            tf.float16, name='input_0'))


tf.saved_model.save(model, 
                    model_path
#                     ,overwrite=True)
                    ,signatures=call)

converter = trt.TrtGraphConverterV2(input_saved_model_dir=model_path,
                                    maximum_cached_engines=100,minimum_segment_size=3,max_workspace_size_bytes=2000000000,
                                    precision_mode=trt.TrtPrecisionMode.FP16,allow_build_at_runtime=True)
converter.convert()
converter.save(output_saved_model_dir=f'{model_path}_FP16')

/usr/local/lib/python3.8/dist-packages/numpy/core/getlimits.py:499: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.8/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.8/dist-packages/numpy/core/getlimits.py:499: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.8/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


INFO:tensorflow:Assets written to: models/simple-trt-model-FP16/1/model.savedmodel/assets
INFO:tensorflow:Linked TensorRT version: (7, 2, 2)
INFO:tensorflow:Loaded TensorRT version: (7, 2, 2)
INFO:tensorflow:Could not find TRTEngineOp_000_000 in TF-TRT cache. This can happen if build() is not called, which means TensorRT engines will be built and cached at runtime.
INFO:tensorflow:Assets written to: models/simple-trt-model-FP16/1/model.savedmodel_FP16/assets


In [8]:
import tensorflow as tf
from tensorflow.python.compiler.tensorrt import trt_convert as trt
# tf.config.optimizer.set_jit(True)

class WrappedModel(tf.Module):
    def __init__(self):
        super(WrappedModel, self).__init__()
        tf.config.optimizer.set_jit(True)
        self.model = tf.keras.Sequential()
        self.model.add(tf.keras.layers.Dense(64, input_shape=(D,)))
        self.model.add(tf.keras.layers.Dense(32))
        self.model.add(tf.keras.layers.Dense(1))
        self.model.compile(optimizer='sgd', loss='mse')
    @tf.function
    def __call__(self, x):
        return self.model(x)
    
model = WrappedModel()
call = model.__call__.get_concrete_function(tf.TensorSpec([None,None], 
                                            tf.float16, name='input_0'))

tf.saved_model.save(model, 
                    model_path_copy
#                     ,overwrite=True)
                    ,signatures=call)


# convert model using trt - keep FP32 precision
# converter = trt.TrtGraphConverterV2(input_saved_model_dir='models/simple-trt-model-FP32/1/model.savedmodel',
#                                     maximum_cached_engines=2,
#                                     precision_mode=trt.TrtPrecisionMode.FP32)
# converter.convert()
# converter.save(output_saved_model_dir=f'{model_path}_FP32')

# convert model using trt - use FP16 precision
converter = trt.TrtGraphConverterV2(input_saved_model_dir=model_path_copy,
                                    maximum_cached_engines=1,
                                    minimum_segment_size=3,
                                    precision_mode=trt.TrtPrecisionMode.FP16)
converter.convert()
converter.save(output_saved_model_dir=f'{model_path_copy}_FP16')

INFO:tensorflow:Assets written to: models/simple-trt-model-FP16-copy/1/model.savedmodel/assets
INFO:tensorflow:Linked TensorRT version: (7, 2, 2)
INFO:tensorflow:Loaded TensorRT version: (7, 2, 2)
INFO:tensorflow:Could not find TRTEngineOp_001_000 in TF-TRT cache. This can happen if build() is not called, which means TensorRT engines will be built and cached at runtime.
INFO:tensorflow:Assets written to: models/simple-trt-model-FP16-copy/1/model.savedmodel_FP16/assets


In [9]:
!curl -v localhost:8000/v2/models/simple-trt-model-FP16/config
   

E0915 17:41:18.743014464     708 backup_poller.cc:133]       Run client channel backup poller: {"created":"@1663263678.742809325","description":"pollset_work","file":"src/core/lib/iomgr/ev_epollex_linux.cc","file_line":320,"referenced_errors":[{"created":"@1663263678.742791653","description":"Bad file descriptor","errno":9,"file":"src/core/lib/iomgr/ev_epollex_linux.cc","file_line":950,"os_error":"Bad file descriptor","syscall":"epoll_wait"}]}
*   Trying ::1:8000...
* TCP_NODELAY set
* connect to ::1 port 8000 failed: Connection refused
*   Trying 127.0.0.1:8000...
* TCP_NODELAY set
* Connected to localhost (127.0.0.1) port 8000 (#0)





* Mark bundle as not supporting multiuse




{"name":"simple-trt-model-FP16","platform":"tensorflow_savedmodel","backend":"tensorflow","version_policy":{"latest":{"num_versions":1}},"max_batch_size":2048,"input":[{"name":"input_0","data_type":"TYPE_FP16","format":"FORMAT_NONE","dims":[1024],"is_shape_tensor":false,"allow_ragged_batch":false,"optional"

In [10]:
configuration = """
name: "simple-trt-model-FP16"
platform: "tensorflow_savedmodel"
max_batch_size: 2048
input [
 {
    name: "input_0"
    data_type: TYPE_FP16
    dims: [ 1024]
  }
]
output {
    name: "output_0"
    data_type: TYPE_FP32
    dims: [ 1 ]
  }
instance_group [
    {
      count: 4
      kind: KIND_GPU
    }
  ]
dynamic_batching { }
response_cache {
  enable: True
  }
optimization { execution_accelerators {
  gpu_execution_accelerator : [ {
    name : "tensorrt"
    parameters { key: "precision_mode" value: "FP16" }}]
}}


"""

with open('models/simple-trt-model-FP16/config.pbtxt', 'w') as file:
    file.write(configuration)

In [11]:
configuration = """
name: "simple-trt-model-FP16-copy"
platform: "tensorflow_savedmodel"
max_batch_size: 2048
input [
 {
    name: "input_0"
    data_type: TYPE_FP16
    dims: [ 1024]
  }
]
output {
    name: "output_0"
    data_type: TYPE_FP32
    dims: [ 1 ]
  }
instance_group [
    {
      count: 4
      kind: KIND_GPU
    }
  ]
dynamic_batching { }
response_cache {
  enable: True
  }
optimization { execution_accelerators {
  gpu_execution_accelerator : [ {
    name : "tensorrt"
    parameters { key: "precision_mode" value: "FP16" }}]
}}


"""

with open('models/simple-trt-model-FP16-copy/config.pbtxt', 'w') as file:
    file.write(configuration)

In [12]:
# tensorflow_savedmodel
# tensorrt_plan
# dynamic_batching {
#   }

In [13]:
# instance_group [ 
#  { 
#      count: 2 
#   }
# ]

## Load Model in Triton Inference Server

In [14]:
!curl -v localhost:8000/v2/health/ready

*   Trying ::1:8000...
* TCP_NODELAY set
* connect to ::1 port 8000 failed: Connection refused
*   Trying 127.0.0.1:8000...
* TCP_NODELAY set
* Connected to localhost (127.0.0.1) port 8000 (#0)





* Mark bundle as not supporting multiuse




* Connection #0 to host localhost left intact


In [15]:
!curl -v localhost:8000/v2/models/simple-trt-model-FP16

*   Trying ::1:8000...
* TCP_NODELAY set
* connect to ::1 port 8000 failed: Connection refused
*   Trying 127.0.0.1:8000...
* TCP_NODELAY set
* Connected to localhost (127.0.0.1) port 8000 (#0)





* Mark bundle as not supporting multiuse




* Connection #0 to host localhost left intact
{"name":"simple-trt-model-FP16","versions":["1"],"platform":"tensorflow_savedmodel","inputs":[{"name":"input_0","datatype":"FP16","shape":[-1,1024]}],"outputs":[{"name":"output_0","datatype":"FP32","shape":[-1,1]}]}

### Shared memory

In [16]:
FLAGS = None

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('-v',
                        '--verbose',
                        action="store_true",
                        required=False,
                        default=False,
                        help='Enable verbose output')
    parser.add_argument('-u',
                        '--url',
                        type=str,
                        required=False,
                        default='localhost:8001',
                        help='Inference server URL. Default is localhost:8001.')

#     FLAGS = parser.parse_args()
    FLAGS, unknown = parser.parse_known_args()

    try:
        triton_client = tritongrpcclient.InferenceServerClient(url=FLAGS.url,
                                                         verbose=FLAGS.verbose)
    except Exception as e:
        print("channel creation failed: " + str(e))
        sys.exit(1)

In [17]:
 # To make sure no shared memory regions are registered with the
    # server.
triton_client.unregister_system_shared_memory()
triton_client.unregister_cuda_shared_memory()

batch1 = (iterator(batch_size))
i = iter(batch1)
batch=next(i)
input_byte_size=batch.size
output_byte_size=input_byte_size

In [18]:
shm_op_handle = shm.create_shared_memory_region("output_data",
                                                "/output_simple-trt-model-FP16",
                                                output_byte_size * 2)

# Register shared memory region for outputs with Triton Server
triton_client.register_system_shared_memory("output_data", "/output_simple-trt-model-FP16",
                                            output_byte_size * 2)



In [19]:
VERBOSE = False
input_name = 'input_0'
input_shape = (batch_size,1024)
input_dtype = 'FP32'
output_name = 'output_0'
model_name = 'simple-trt-model-FP16'
model_name_copy = 'simple-trt-model-FP16-copy'
url = 'localhost:8000'
model_version = '1'
request_compression_algorithm='gzip'
response_compression_algorithm='gzip'

In [20]:
input0 = tritongrpcclient.InferInput(input_name, batch.shape, 'FP16')
input0.set_data_from_numpy(batch)                        
output = tritongrpcclient.InferRequestedOutput(output_name)


In [21]:
print(type(input0))
input0_data = np.arange(start=0, stop=16, dtype=np.int16)

<class 'tritonclient.grpc.InferInput'>


### # Create shared memory region for output and store shared memory handle

In [22]:
import numpy
# Put input data values into shared memory
def __init__(self):
    self.coordinate_row = []

def add(self, input):
    self.coordinate_row.append(input)

def __len__(self):
    return len(self.coordinate_row)


# Create shared memory region for input and store shared memory handle
shm_ip_handle = shm.create_shared_memory_region("input_data",
                                                "/input_simple-trt-model-FP16",
                                                input_byte_size * 2)


shm.set_shared_memory_region(shm_ip_handle, [input0_data])
# shm.set_shared_memory_region(shm_ip_handle, [numpy.asarray(input0)],
#                              offset=input_byte_size)






# Register shared memory region for inputs with Triton Server
triton_client.register_system_shared_memory("input_data", "/input_simple-trt-model-FP16",
                                            input_byte_size * 2)

# Set the parameters to use data from shared memory
inputs = []
inputs.append(grpcclient.InferInput('INPUT0', [1, 2048], "FP16"))
inputs[-1].set_shared_memory("input_data", input_byte_size)

# inputs.append(grpcclient.InferInput('INPUT1', [1, 16], "INT32"))
# inputs[-1].set_shared_memory("input_data",
#                              input_byte_size,
#                              offset=input_byte_size)

outputs = []
outputs.append(grpcclient.InferRequestedOutput('OUTPUT0'))
outputs[-1].set_shared_memory("output_data", output_byte_size)

# outputs.append(grpcclient.InferRequestedOutput('OUTPUT1'))
# outputs[-1].set_shared_memory("output_data",
#                               output_byte_size,
#                               offset=output_byte_size)

results = triton_client.infer(model_name=model_name,
                              inputs=inputs,
                              outputs=outputs)

outputs = []
outputs.append(grpcclient.InferRequestedOutput('OUTPUT0'))
outputs[-1].set_shared_memory("output_data", output_byte_size)

# outputs.append(grpcclient.InferRequestedOutput('OUTPUT1'))
# outputs[-1].set_shared_memory("output_data",
#                               output_byte_size,
#                               offset=output_byte_size)

results = triton_client.infer(model_name=model_name,
                              inputs=inputs,
                              outputs=outputs)


InferenceServerException: [StatusCode.INVALID_ARGUMENT] unexpected inference output 'OUTPUT0' for model 'simple-trt-model-FP16'

In [23]:
!curl -v localhost:8000/v2/models/simple-trt-model-FP16-copy

*   Trying ::1:8000...
* TCP_NODELAY set
* connect to ::1 port 8000 failed: Connection refused
*   Trying 127.0.0.1:8000...
* TCP_NODELAY set
* Connected to localhost (127.0.0.1) port 8000 (#0)





* Mark bundle as not supporting multiuse




* Connection #0 to host localhost left intact
{"name":"simple-trt-model-FP16-copy","versions":["1"],"platform":"tensorflow_savedmodel","inputs":[{"name":"input_0","datatype":"FP16","shape":[-1,1024]}],"outputs":[{"name":"output_0","datatype":"FP32","shape":[-1,1]}]}

## Send Inference Request to Server

In [24]:
import tritonclient.http as tritonhttpclient
from tritonclient.utils import triton_to_np_dtype

In [25]:
VERBOSE = False
input_name = 'input_0'
input_shape = (batch_size,1024)
input_dtype = 'FP32'
output_name = 'output_0'
model_name = 'simple-trt-model-FP16'
model_name_copy = 'simple-trt-model-FP16-copy'
url = 'localhost:8000'
model_version = '1'
request_compression_algorithm='gzip'
response_compression_algorithm='gzip'

In [26]:
print(model_name,model_version,url)

simple-trt-model-FP16 1 localhost:8000


#### We'll instantiate our client triton_client using the tritonhttpclient.InferenceServerClient class access the model metadata with the .get_model_metadata() method as well as get our model configuration with the get_model_config() method.

In [27]:
triton_client = tritonhttpclient.InferenceServerClient(url=url, verbose=VERBOSE)
model_metadata = triton_client.get_model_metadata(model_name=model_name, model_version=model_version)
model_config = triton_client.get_model_config(model_name=model_name, model_version=model_version)

In [28]:
triton_client = tritonhttpclient.InferenceServerClient(url=url, verbose=VERBOSE)
model_metadata_copy = triton_client.get_model_metadata(model_name=model_name_copy, model_version=model_version)
model_config_copy = triton_client.get_model_config(model_name=model_name_copy, model_version=model_version)

In [29]:
batch1 = (iterator(batch_size))


In [30]:
# batch1 = next(batch)
i = iter(batch1)

In [31]:
# print(next(i))

In [32]:
batch=next(i)

In [33]:
print(batch.shape,",",batch.dtype)

(2048, 1024) , float16


In [34]:
print(input_shape,",",input_dtype)

(2048, 1024) , FP32


#### We'll instantiate a placeholder for our input data using the input name, shape, and data type expected. We'll set the data of the input to be the NumPy array representation of our goldfish image. We'll also instantiate a placeholder for our output data using just the output name.

#### Lastly, we'll submit our input to the Triton Inference Server using the triton_client.infer() method, specifying our model name, model version, inputs, and outputs and convert our result to a NumPy array.

In [35]:
input0 = tritonhttpclient.InferInput(input_name, batch.shape, 'FP16')
# input0 = tritonhttpclient.InferInput(input_name, input_shape, input_dtype)
input0.set_data_from_numpy(batch, binary_data=True)

output = tritonhttpclient.InferRequestedOutput(output_name, binary_data=False)
# response = triton_client.infer(model_name, model_version=model_version, 
#                                inputs=[input0], outputs=[output])
# logits = response.as_numpy(output_name)
# logits = np.asarray(logits, dtype=np.float32)

In [36]:
# response_copy = triton_client.infer(model_name_copy, model_version=model_version, 
#                                inputs=[input0], outputs=[output])
# logits = response.as_numpy(output_name)
# logits = np.asarray(logits, dtype=np.float32)

### HTTP vs. gRPC

#### Let's submit 1000 requests

#### http

In [37]:
# start_time = time.time()
# requests = []
# request_count = 1000
# for i in tqdm(range(request_count)):
#     requests.append(triton_http_client.infer(model_name, model_version=model_version, 
#                                              inputs=[input0], outputs=[output]))
# end_time = time.time()

In [38]:

# print('Average Latency: ~{} seconds'.format((end_time - start_time) / request_count))
# print('Average Throughput: ~{} examples / second'.format(batch_size * request_count / (end_time - start_time)))

#### http Compression

In [39]:

# start_time = time.time()
# requests = []
# request_count = 1000
# for i in tqdm(range(request_count)):
#     requests.append(triton_http_client.infer(
#         model_name, model_version=model_version, 
#         inputs=[input0], outputs=[output],
#         request_compression_algorithm=bytes(request_compression_algorithm, 'utf-8'),
#         response_compression_algorithm=bytes(response_compression_algorithm, 'utf-8')))
# end_time = time.time()

In [40]:

# print('Average Latency: ~{} seconds'.format((end_time - start_time) / request_count))
# print('Average Throughput: ~{} examples / second'.format(batch_size * request_count / (end_time - start_time)))

#### gRPC

##### Clients can communicate with Triton using either an HTTP/REST or GRPC protocol, or by a C API. Most people are familiar with HTTP, which is the backbone of the internet. gRPC is a newer, open source remote procedure call system initially developed at Google in 2015 that uses HTTP/2 for transport and Protocol Buffers as the interface description language. It is highly efficient and using it is very easy.

##### Below, we use the tritonclient.grpc module to instantiate new InferInput and InferRequestedOutput objects, and our tritonclient.grpc.InferenceServerClient instance to send 10000 requests of batch size 1 to our dynamic-batching-model. We can immediately see that just using a slightly different protocol can have an enormous impact on latency and throughput!

In [41]:
# !python grpc_image_client.py -i grpc -u localhost:8001 -m /v2/models/simple-trt-model-FP32 -s INCEPTION batch 
# !python grpc_image_client.py -m simple-trt-model-FP32 -s INCEPTION -c 3 batch
# !grpc -u localhost:8001 -m simple-trt-model-FP32 -s INCEPTION batch
# input_saved_model_dir

In [42]:
input0 = tritongrpcclient.InferInput(input_name, batch.shape, 'FP16')
input0.set_data_from_numpy(batch)
output = tritongrpcclient.InferRequestedOutput(output_name)


### # Create shared memory region for output and store shared memory handle

In [43]:
# import numpy
# # Put input data values into shared memory
# def __init__(self):
#     self.coordinate_row = []

# def add(self, input):
#     self.coordinate_row.append(input)

# def __len__(self):
#     return len(self.coordinate_row)

    
# shm.set_shared_memory_region(shm_ip_handle, [input0])
# # shm.set_shared_memory_region(shm_ip_handle, [numpy.asarray(input0)],
# #                              offset=input_byte_size)

# # Register shared memory region for inputs with Triton Server
# triton_client.register_system_shared_memory("input_data", "/input_simple",
#                                             input_byte_size * 2)

# # Set the parameters to use data from shared memory
# inputs = []
# inputs.append(grpcclient.InferInput('INPUT0', [1, 16], "FP16"))
# inputs[-1].set_shared_memory("input_data", input_byte_size)

# # inputs.append(grpcclient.InferInput('INPUT1', [1, 16], "INT32"))
# # inputs[-1].set_shared_memory("input_data",
# #                              input_byte_size,
# #                              offset=input_byte_size)

# outputs = []
# outputs.append(grpcclient.InferRequestedOutput('OUTPUT0'))
# outputs[-1].set_shared_memory("output_data", output_byte_size)

# outputs.append(grpcclient.InferRequestedOutput('OUTPUT1'))
# outputs[-1].set_shared_memory("output_data",
#                               output_byte_size,
#                               offset=output_byte_size)

# results = triton_client.infer(model_name=model_name,
#                               inputs=inputs,
#                               outputs=outputs)

# outputs = []
# outputs.append(grpcclient.InferRequestedOutput('OUTPUT0'))
# outputs[-1].set_shared_memory("output_data", output_byte_size)

# outputs.append(grpcclient.InferRequestedOutput('OUTPUT1'))
# outputs[-1].set_shared_memory("output_data",
#                               output_byte_size,
#                               offset=output_byte_size)

# results = triton_client.infer(model_name=model_name,
#                               inputs=inputs,
#                               outputs=outputs)


### blocking in gRPC

In [44]:
# start_time = time.time()
# requests = []
# request_count = 1000
# for i in tqdm(range(request_count)):
    
#     if (i%2 ==0):
#         requests.append(triton_grpc_client.infer(model_name, model_version=model_version, 
#                                              inputs=[input0], outputs=[output]))
        
#     else:
#         requests.append(triton_grpc_client.infer(model_name_copy, model_version=model_version, 
#                                              inputs=[input0], outputs=[output]))
    
    
# end_time = time.time()

In [45]:
# requests[0].__dict__

In [46]:

# print('Average Latency: ~{} seconds'.format((end_time - start_time) / request_count))
# print('Average Throughput: ~{} examples / second'.format(batch_size * request_count / (end_time - start_time)))

### Asynchronous Calling in gRPC

In [47]:
from functools import partial
import sys

if sys.version_info >= (3, 0):
    import queue
else:
    import Queue as queue

class UserData:

    def __init__(self):
        self._completed_requests = queue.Queue()
        
def __init__(self):
    self._completed_requests = queue.Queue()
def completion_callback(user_data, result, error):
    # passing error raise and handling out
    user_data._completed_requests.put((result, error))

user_data = UserData()



In [48]:
models = [model_name, model_name_copy]
start_time = time.time()
async_requests = []
responses = []
request_count = 1000
for i in tqdm(range(request_count)):
        responses = []
        sent_count = 1
        async_requests.append(triton_grpc_client.async_infer(model_name=models[i%2], inputs=[input0],
                                             callback=partial(completion_callback, user_data),
                                                 outputs=[output]))

processed_count = 0
while processed_count < sent_count:
    (results, error) = user_data._completed_requests.get()
    processed_count += 1
    if error is not None:
        print("inference failed: " + str(error))
        sys.exit(1)
    responses.append(user_data) 

end_time = time.time()

100%|██████████| 1000/1000 [00:04<00:00, 202.31it/s]


In [49]:
# results[0].__dict__

In [50]:
print('Average Latency: ~{} seconds'.format((end_time - start_time) / request_count))
print('Average Throughput: ~{} examples / second'.format(batch_size * request_count / (end_time - start_time)))

Average Latency: ~0.005370535850524902 seconds
Average Throughput: ~381339.97370110353 examples / second


In [51]:
from functools import partial


results = []

def callback(user_data, result, error):
    if error:
        user_data.append(error)
    else:
        user_data.append(result)

In [52]:
models = [model_name, model_name_copy]

start_time = time.time()
async_requests = []
request_count = 1000
for i in tqdm(range(request_count)):
        async_requests.append(triton_grpc_client.async_infer(model_name=models[i%2], inputs=[input0],
                                                     callback=partial(callback, results),
                                                         outputs=[output]))
end_time = time.time()

100%|██████████| 1000/1000 [00:04<00:00, 233.37it/s]


In [53]:
print('Average Latency: ~{} seconds'.format((end_time - start_time) / request_count))
print('Average Throughput: ~{} examples / second'.format(batch_size * request_count / (end_time - start_time)))

Average Latency: ~0.004293572902679443 seconds
Average Throughput: ~476992.0172362572 examples / second


In [54]:
# results[0].__dict__

In [55]:
print(batch.shape,",",batch.dtype)

(2048, 1024) , float16


#### Applying gRPC Compression

In [56]:
# start_time = time.time()
# requests = []
# request_count = 1000
# for i in tqdm(range(request_count)):
#     requests.append(triton_grpc_client.infer(model_name, model_version=model_version, 
#                                              inputs=[input0], outputs=[output],
#                                             compression_algorithm="gzip"))
# end_time = time.time()

In [57]:
# print('Average Latency: ~{} seconds'.format((end_time - start_time) / request_count))
# print('Average Throughput: ~{} examples / second'.format(batch_size * request_count / (end_time - start_time)))

In [58]:
# start_time = time.time()
# requests = []
# request_count = 1000
# for i in tqdm(range(request_count)):
#     requests.append(triton_grpc_client.infer(model_name, model_version=model_version, 
#                                              inputs=[input0], outputs=[output],
#                                             compression_algorithm="deflate"))
# end_time = time.time()

In [59]:
# print('Average Latency: ~{} seconds'.format((end_time - start_time) / request_count))
# print('Average Throughput: ~{} examples / second'.format(batch_size * request_count / (end_time - start_time)))

In [60]:
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)


###### 